## ***RNN***
 - Recurent Neural Network
 - super important for sequence data like text, time-series, speech, and more.
 ### Problem:
 - Standard feed-forward neural networks can't remember past inputs — they treat each input independently.

### Solution:
- RNNs introduce a hidden state that carries information from previous steps → giving the network "memory."

### RNN Mathematics

At time step **t**:

- **Input:**  
  $$x_t$$

- **Hidden state:**  
  $$
  h_t = f(W_{xh} \, x_t + W_{hh} \, h_{t-1} + b_h)
  $$

- **Output:**  
  $$
  y_t = W_{hy} \, h_t + b_y
  $$

Where \(f\) is usually **tanh** or **ReLU**.


### Key Concepts

- Hidden State: Carries information through time.

- Unrolling: RNN is applied repeatedly for each time step.

- Backpropagation Through Time (BPTT): How RNNs learn (but suffers from vanishing/exploding gradients).

- Limitations: Struggles with very long sequences → solved by LSTM and GRU.

### PyTorch Modules :

- `nn.RNN` → Vanilla RNN

- `nn.LSTM` → Long Short-Term Memory

- `nn.GRU` → Gated Recurrent Unit

### Predict the next number in a sequence :

In [1]:
import torch
import torch.nn as nn

seq = torch.tensor([[0, 1, 2, 3]], dtype=torch.float32)  # shape [1, 4]
target = torch.tensor([[1, 2, 3, 4]], dtype=torch.float32)  # shifted by 1

# Reshape: RNN expects (seq_len, batch, input_size)
seq = seq.view(4, 1, 1)      # 4 timesteps, batch=1, input_size=1
target = target.view(4, 1, 1)

In [2]:
rnn = nn.RNN(input_size=1, hidden_size=10, num_layers=1)
linear = nn.Linear(10, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(list(rnn.parameters()) + list(linear.parameters()), lr=0.01)

In [3]:
for epoch in range(300):
    optimizer.zero_grad()

    h0 = torch.zeros(1, 1, 10)  # initial hidden state (num_layers, batch_size, hidden_size )
    out, hn = rnn(seq, h0)      # out shape: [4, 1, 10]
    pred = linear(out)          # map to output size 1

    loss = criterion(pred, target)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 50 == 0:
        print(f"Epoch [{epoch+1}/300], Loss: {loss.item():.6f}")

Epoch [50/300], Loss: 0.275286
Epoch [100/300], Loss: 0.011601
Epoch [150/300], Loss: 0.000287
Epoch [200/300], Loss: 0.000003
Epoch [250/300], Loss: 0.000000
Epoch [300/300], Loss: 0.000000


### **Long Short-Term Memory (LSTM)**

* Designed to solve the **vanishing gradient problem**.
* Uses **gates** (input, forget, output) to control what information to keep or discard.
* Maintains two states:
  * **Hidden state:** \(h_t\) (short-term memory)
  * **Cell state:** \(c_t\) (long-term memory)

**Mathematics:**

At each time step:

$$
\begin{aligned}
f_t &= \sigma(W_f x_t + U_f h_{t-1} + b_f) &\quad \text{(Forget Gate)}\\[6pt]
i_t &= \sigma(W_i x_t + U_i h_{t-1} + b_i) &\quad \text{(Input Gate)}\\[6pt]
\tilde{c}_t &= \tanh(W_c x_t + U_c h_{t-1} + b_c) &\quad \text{(Candidate State)}\\[6pt]
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t &\quad \text{(Cell Update)}\\[6pt]
o_t &= \sigma(W_o x_t + U_o h_{t-1} + b_o) &\quad \text{(Output Gate)}\\[6pt]
h_t &= o_t \odot \tanh(c_t) &\quad \text{(Hidden State Update)}
\end{aligned}
$$

Where:  
- ( $\sigma$) = Sigmoid activation  

- ( $\odot$) = Element-wise multiplication  


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# Creating dataset
seq_len = 5
data = torch.arange(1, 21, dtype=torch.float32)

def create_sequences(data, seq_len):
    xs, ys = [], []
    for i in range(len(data) - seq_len):
        xs.append(data[i:i+seq_len])
        ys.append(data[i+seq_len])
    return torch.stack(xs), torch.stack(ys)


In [4]:
X, y = create_sequences(data, seq_len)
X = X.unsqueeze(-1)  # (batch, seq_len, input_size)
y = y.unsqueeze(-1)

print(f"[LSTM] Dataset shapes: X={X.shape}, y={y.shape}")

[LSTM] Dataset shapes: X=torch.Size([15, 5, 1]), y=torch.Size([15, 1])


In [5]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=16, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # last time-step output
        out = self.fc(out)
        return out

In [6]:
model = LSTMModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


In [7]:
for epoch in range(200):
    optimizer.zero_grad()
    y_pred = model(X)
    loss = criterion(y_pred, y)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 50 == 0:
        print(f"[LSTM] Epoch {epoch+1}: Loss = {loss.item():.4f}")

[LSTM] Epoch 50: Loss = 42.2507
[LSTM] Epoch 100: Loss = 18.8856
[LSTM] Epoch 150: Loss = 5.9620
[LSTM] Epoch 200: Loss = 1.7002


In [8]:
with torch.no_grad():
    predictions = model(X)
print("\n[LSTM] First 5 Predictions vs Actual:")
print("Pred:", predictions[:5].squeeze().numpy())
print("True:", y[:5].squeeze().numpy())


[LSTM] First 5 Predictions vs Actual:
Pred: [ 6.059236   6.9531884  7.938493   8.982776  10.043319 ]
True: [ 6.  7.  8.  9. 10.]


### **Gated Recurrent Unit (GRU)**

* A **simplified version of LSTM** (fewer parameters, faster training).
* Combines forget and input gates into a single **update gate**.
* Maintains only **hidden state \(h_t\)** (no separate cell state).

**Mathematics:**

$$
\begin{aligned}
z_t &= \sigma(W_z x_t + U_z h_{t-1}) &\quad \text{(Update Gate)}\\[6pt]
r_t &= \sigma(W_r x_t + U_r h_{t-1}) &\quad \text{(Reset Gate)}\\[6pt]
\tilde{h}_t &= \tanh(W_h x_t + U_h (r_t \odot h_{t-1})) &\quad \text{(Candidate State)}\\[6pt]
h_t &= (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t &\quad \text{(Final Hidden State)}
\end{aligned}
$$


In [9]:
X, y = create_sequences(data, seq_len)
X = X.unsqueeze(-1)
y = y.unsqueeze(-1)

print(f"[GRU] Dataset shapes: X={X.shape}, y={y.shape}")

[GRU] Dataset shapes: X=torch.Size([15, 5, 1]), y=torch.Size([15, 1])


In [10]:
class GRUModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=16, num_layers=1):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]  # last time-step output
        out = self.fc(out)
        return out

In [11]:
model = GRUModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [12]:
for epoch in range(200):
    optimizer.zero_grad()
    y_pred = model(X)
    loss = criterion(y_pred, y)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 50 == 0:
        print(f"[GRU] Epoch {epoch+1}: Loss = {loss.item():.4f}")

[GRU] Epoch 50: Loss = 35.3919
[GRU] Epoch 100: Loss = 16.9251
[GRU] Epoch 150: Loss = 5.1798
[GRU] Epoch 200: Loss = 1.5481


In [13]:
with torch.no_grad():
    predictions = model(X)
print("\n[GRU] First 5 Predictions vs Actual:")
print("Pred:", predictions[:5].squeeze().numpy())
print("True:", y[:5].squeeze().numpy())


[GRU] First 5 Predictions vs Actual:
Pred: [5.9335766 7.12894   8.114189  9.027187  9.966831 ]
True: [ 6.  7.  8.  9. 10.]
